# 03 — Leakage Audit (SF-038)

Component 3 (Learning Insight & Early Warning System) — Grade 2-3 mathematics dyscalculia,
synthetic research data.

**Purpose of this notebook:** this is a GATE. NB04–NB07 do not run if it fails. The job here is to
try hard to *break* the feature set built in NB02, not to certify it. Every feature space NB05 could
plausibly consume gets its own SF-038 run: the full engineered matrix, the NB01-original 19-feature
raw block, and the within-`t` standardised arm.

The audit labels (`t3_audit_labels_seed20260828_n300.csv` — `student_id`, `population`,
`dd_subtype`) are quarantined. They are merged in *only* for this audit and are asserted absent from
every saved artefact at the end of this notebook.

References: `docs/08_LEAKAGE_AND_SAFETY_RULES.md` (SF-038, rules 3-F1/F2/F3/F8/F9, CL-3),
`docs/06_FEATURE_SPECIFICATION.md` §4 (slope censoring),
`dev_docs/notebook_findings/02_preprocessing_findings.md` §10 (task list this notebook executes).

Does **not** model the target, tune hyperparameters, or cluster — that is NB04/NB05.

In [1]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score, f1_score, roc_auc_score, confusion_matrix,
)

from nbconfig import *

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def savefig(fig, name):
    path = FIGURES_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print("saved:", path)

def savetable(df_, name, index=True):
    path = TABLES_DIR / f"{name}.csv"
    df_.to_csv(path, index=index)
    print("saved:", path)

print("Random seed:", RANDOM_SEED)

Random seed: 20260828


## 1. Load feature spaces and the quarantined audit labels

Three spaces, per task §B:

- **(i)** the full engineered matrix (`24_engineered_feature_matrix.csv`, 2,700 × 62)
- **(ii)** the NB01-original 19 non-degenerate model-role features, taken from the *raw* training
  table (NB02 dropped 4 of the 23 as degenerate — `guess_guard_rate`, `engagement_consistency`,
  `hint_avoidance`, `stagnant_skill_count` — and 3 more were dropped from the engineered matrix
  entirely as dead columns, so this space is pulled from the pre-engineering table, not the matrix)
- **(iii)** the within-`t` standardised block (`21_within_t_standardised_optional.csv`, 2,700 × 8)
  — NB05 arm B

`t3_audit_labels` is learner-level (`student_id`, `population`, `dd_subtype`) and is merged onto
each space by `student_id` only for this audit.

In [2]:
audit = pd.read_csv(AUDIT_LABELS_PATH)
assert audit.shape == (300, 3)
assert set(audit.columns) == {"student_id", "population", "dd_subtype"}
print("Audit labels:", audit.shape)
print(audit["population"].value_counts(normalize=True).round(4))
print(audit["dd_subtype"].value_counts(normalize=True).round(4))

engineered = pd.read_csv(TABLES_DIR / "24_engineered_feature_matrix.csv")
assert engineered.shape == (2700, 62)
print("\nEngineered matrix:", engineered.shape)

within_t = pd.read_csv(TABLES_DIR / "21_within_t_standardised_optional.csv")
assert within_t.shape == (2700, 8)
print("Within-t standardised:", within_t.shape)

raw = pd.read_csv(TRAINING_TABLE_PATH)
assert raw.shape[0] == 2700
print("Raw training table:", raw.shape)
assert "population" not in raw.columns and "dd_subtype" not in raw.columns
assert "population" not in engineered.columns and "dd_subtype" not in engineered.columns
assert "population" not in within_t.columns and "dd_subtype" not in within_t.columns
print("\nConfirmed: no space carries population/dd_subtype prior to the audit merge below.")

Audit labels: (300, 3)
population
TD     0.7433
MLD    0.1633
DD     0.0933
Name: proportion, dtype: float64
dd_subtype
none    1.0
Name: proportion, dtype: float64

Engineered matrix: (2700, 62)
Within-t standardised: (2700, 8)
Raw training table: (2700, 52)

Confirmed: no space carries population/dd_subtype prior to the audit merge below.


In [3]:
# NB01-original 19 non-degenerate model-role features (of the 23 in MODEL_FEATURES),
# pulled from the raw training table — 3 of these 19 (rt_change_ratio, rt_variability,
# game_latency_change_ratio, hint_intensity are 4 -- listed below) were engineered away /
# replaced by indicators in NB02's matrix, so this space must come from `raw`, not `engineered`.
NB01_DEGENERATE = ["guess_guard_rate", "engagement_consistency", "hint_avoidance", "stagnant_skill_count"]
NB01_19_FEATURES = [f for f in MODEL_FEATURES if f not in NB01_DEGENERATE]
assert len(NB01_19_FEATURES) == 19, len(NB01_19_FEATURES)
print("NB01-original 19 features:")
print(NB01_19_FEATURES)
for f in NB01_19_FEATURES:
    assert f in raw.columns, f"{f} missing from raw training table"


NB01-original 19 features:
['score_trend_slope', 'score_avg_recent', 'mastery_progression_rate', 'number_sense_mastery', 'weakest_skill_persistence', 'rt_change_ratio', 'rt_variability', 'rt_accuracy_tradeoff', 'game_latency_change_ratio', 'error_recurrence_rate', 'hint_intensity', 'engagement_decline_rate', 'frustration_mean', 'frustration_slope', 'difficulty_floor_hits', 'sessions_observed', 'b_sessions_observed', 'c_sessions_observed', 'slope_censored_count']


In [4]:
def merge_audit(space_df, id_col="student_id"):
    """Merge quarantined labels onto a feature space for this audit only."""
    m = space_df.merge(audit, on=id_col, how="left", validate="many_to_one")
    assert m["population"].notnull().all(), "unmatched student_id against audit labels"
    return m

engineered_a = merge_audit(engineered)
raw19_a = merge_audit(raw[["student_id", "t", "split"] + NB01_19_FEATURES])
within_t_a = merge_audit(within_t)

print(engineered_a.shape, raw19_a.shape, within_t_a.shape)

(2700, 64) (2700, 24) (2700, 10)


## 2. SF-038 audit function

Depth-4 `DecisionTreeClassifier`, run under two CV schemes so the size of the grouping effect is
visible directly:

- **Grouped** (`StratifiedGroupKFold(groups=student_id)`) — the correct scheme. `population` is a
  learner-level constant; a plain K-fold puts a learner's other 8 snapshots in the training fold,
  so the tree can memorise the learner rather than learn a feature–population relationship. That
  would inflate the score and could fail the gate spuriously.
- **Ungrouped** (`StratifiedKFold`) — reported purely to show the size of that inflation, never used
  for the gate verdict.

Reported for both: balanced accuracy (**the gate**), raw accuracy, majority-class baseline, macro-F1,
one-vs-rest macro ROC-AUC, and the confusion matrix. A stratified-dummy baseline is run through the
same grouped folds for direct comparison.

In [5]:
def _run_cv(X, y, groups, splitter, use_groups, estimator_factory, classes_sorted):
    n = len(y)
    oof_pred = np.empty(n, dtype=object)
    oof_proba = np.zeros((n, len(classes_sorted)))
    fold_bal_acc = []
    split_iter = splitter.split(X, y, groups) if use_groups else splitter.split(X, y)
    for tr_idx, te_idx in split_iter:
        est = estimator_factory()
        est.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred = est.predict(X.iloc[te_idx])
        proba = est.predict_proba(X.iloc[te_idx])
        oof_pred[te_idx] = pred
        for ci, cls in enumerate(est.classes_):
            col = int(np.where(classes_sorted == cls)[0][0])
            oof_proba[te_idx, col] = proba[:, ci]
        fold_bal_acc.append(balanced_accuracy_score(y.iloc[te_idx], pred))
    return oof_pred, oof_proba, fold_bal_acc


def sf038_audit(X, y, groups, n_splits=5, random_state=RANDOM_SEED, verbose=True):
    """Runs the SF-038 gate: depth-4 tree, grouped + ungrouped CV, dummy + majority baselines."""
    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    groups = pd.Series(groups).reset_index(drop=True)
    classes_sorted = np.sort(y.unique())

    tree_factory = lambda: DecisionTreeClassifier(max_depth=4, random_state=random_state)
    dummy_factory = lambda: DummyClassifier(strategy="stratified", random_state=random_state)

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    tree_grouped_pred, tree_grouped_proba, tree_grouped_folds = _run_cv(
        X, y, groups, sgkf, True, tree_factory, classes_sorted)
    tree_ungrouped_pred, tree_ungrouped_proba, tree_ungrouped_folds = _run_cv(
        X, y, groups, skf, False, tree_factory, classes_sorted)
    dummy_grouped_pred, dummy_grouped_proba, dummy_grouped_folds = _run_cv(
        X, y, groups, sgkf, True, dummy_factory, classes_sorted)

    majority_class = y.value_counts().idxmax()
    majority_pred = np.full(len(y), majority_class, dtype=object)

    def metrics_block(pred, proba, fold_accs, label):
        bal_acc = balanced_accuracy_score(y, pred)
        acc = accuracy_score(y, pred)
        f1_macro = f1_score(y, pred, average="macro")
        try:
            auc = roc_auc_score(y, proba, multi_class="ovr", average="macro", labels=classes_sorted)
        except ValueError:
            auc = np.nan
        cm = confusion_matrix(y, pred, labels=classes_sorted)
        if verbose:
            print(f"--- {label} ---")
            print(f"balanced_accuracy = {bal_acc:.3f}  (fold range {min(fold_accs):.3f}-{max(fold_accs):.3f})"
                  if fold_accs else f"balanced_accuracy = {bal_acc:.3f}")
            print(f"accuracy = {acc:.3f}  macro_f1 = {f1_macro:.3f}  ovr_macro_auc = {auc:.3f}")
        return {
            "label": label, "balanced_accuracy": bal_acc, "accuracy": acc,
            "macro_f1": f1_macro, "ovr_macro_auc": auc,
            "confusion_matrix": cm.tolist(), "classes": classes_sorted.tolist(),
            "fold_balanced_accuracies": fold_accs,
        }

    results = {
        "tree_grouped": metrics_block(tree_grouped_pred, tree_grouped_proba, tree_grouped_folds, "tree (grouped CV)"),
        "tree_ungrouped": metrics_block(tree_ungrouped_pred, tree_ungrouped_proba, tree_ungrouped_folds, "tree (ungrouped CV)"),
        "dummy_grouped": metrics_block(dummy_grouped_pred, dummy_grouped_proba, dummy_grouped_folds, "stratified dummy (grouped CV)"),
        "majority_baseline": metrics_block(majority_pred, None, [], "majority-class baseline") if False else None,
    }
    maj_bal_acc = balanced_accuracy_score(y, majority_pred)
    maj_acc = accuracy_score(y, majority_pred)
    if verbose:
        print(f"--- majority-class baseline ({majority_class}) ---")
        print(f"balanced_accuracy = {maj_bal_acc:.3f}  accuracy = {maj_acc:.3f}")
    results["majority_baseline"] = {
        "label": f"majority-class baseline ({majority_class})",
        "balanced_accuracy": maj_bal_acc, "accuracy": maj_acc,
        "macro_f1": np.nan, "ovr_macro_auc": np.nan,
        "confusion_matrix": None, "classes": classes_sorted.tolist(), "fold_balanced_accuracies": [],
    }
    return results

print("sf038_audit ready.")

sf038_audit ready.


## 3. Space (i) — full engineered matrix vs `population`

**GATE: balanced accuracy < 0.70.**

In [6]:
feat_cols_engineered = [c for c in engineered.columns if c not in ("student_id", "t", "split", "future_stagnant_count")]
print(f"{len(feat_cols_engineered)} feature columns in space (i)")

X_i = engineered_a[feat_cols_engineered].copy()
# tree cannot handle NaN directly in this sklearn build check -> impute simply for the audit only
X_i = X_i.fillna(X_i.median(numeric_only=True))
y_i = engineered_a["population"]
groups_i = engineered_a["student_id"]

res_i = sf038_audit(X_i, y_i, groups_i)
gate_i = res_i["tree_grouped"]["balanced_accuracy"]
print(f"\nGATE (space i, grouped): {gate_i:.3f} vs 0.70 threshold, vs 0.333 chance, "
      f"vs dummy {res_i['dummy_grouped']['balanced_accuracy']:.3f}")
print(f"PASS" if gate_i < 0.70 else "FAIL")

58 feature columns in space (i)


--- tree (grouped CV) ---
balanced_accuracy = 0.352  (fold range 0.342-0.362)
accuracy = 0.717  macro_f1 = 0.338  ovr_macro_auc = 0.465
--- tree (ungrouped CV) ---
balanced_accuracy = 0.363  (fold range 0.346-0.377)
accuracy = 0.747  macro_f1 = 0.347  ovr_macro_auc = 0.577
--- stratified dummy (grouped CV) ---
balanced_accuracy = 0.341  (fold range 0.327-0.353)
accuracy = 0.595  macro_f1 = 0.341  ovr_macro_auc = 0.505
--- majority-class baseline (TD) ---
balanced_accuracy = 0.333  accuracy = 0.743

GATE (space i, grouped): 0.352 vs 0.70 threshold, vs 0.333 chance, vs dummy 0.341
PASS


## 4. Space (ii) — NB01-original 19 raw features vs `population`

In [7]:
X_ii = raw19_a[NB01_19_FEATURES].copy()
X_ii = X_ii.fillna(X_ii.median(numeric_only=True))
y_ii = raw19_a["population"]
groups_ii = raw19_a["student_id"]

res_ii = sf038_audit(X_ii, y_ii, groups_ii)
gate_ii = res_ii["tree_grouped"]["balanced_accuracy"]
print(f"\nGATE (space ii, grouped): {gate_ii:.3f} vs 0.70 threshold, vs 0.333 chance, "
      f"vs dummy {res_ii['dummy_grouped']['balanced_accuracy']:.3f}")
print("PASS" if gate_ii < 0.70 else "FAIL")

--- tree (grouped CV) ---
balanced_accuracy = 0.337  (fold range 0.313-0.353)
accuracy = 0.717  macro_f1 = 0.310  ovr_macro_auc = 0.469
--- tree (ungrouped CV) ---
balanced_accuracy = 0.352  (fold range 0.346-0.362)
accuracy = 0.739  macro_f1 = 0.330  ovr_macro_auc = 0.589
--- stratified dummy (grouped CV) ---
balanced_accuracy = 0.341  (fold range 0.327-0.353)
accuracy = 0.595  macro_f1 = 0.341  ovr_macro_auc = 0.505
--- majority-class baseline (TD) ---
balanced_accuracy = 0.333  accuracy = 0.743

GATE (space ii, grouped): 0.337 vs 0.70 threshold, vs 0.333 chance, vs dummy 0.341
PASS


## 5. Space (iii) — within-`t` standardised block vs `population`

Per NB02 §6, this arm may carry NB05 (best like-for-like MI). It needs its own clearance.

In [8]:
feat_cols_within = [c for c in within_t.columns if c not in ("student_id", "t", "split")]
print(feat_cols_within)

X_iii = within_t_a[feat_cols_within].copy()
X_iii = X_iii.fillna(X_iii.median(numeric_only=True))
y_iii = within_t_a["population"]
groups_iii = within_t_a["student_id"]

res_iii = sf038_audit(X_iii, y_iii, groups_iii)
gate_iii = res_iii["tree_grouped"]["balanced_accuracy"]
print(f"\nGATE (space iii, grouped): {gate_iii:.3f} vs 0.70 threshold, vs 0.333 chance, "
      f"vs dummy {res_iii['dummy_grouped']['balanced_accuracy']:.3f}")
print("PASS" if gate_iii < 0.70 else "FAIL")

['score_trend_slope_within_t_z', 'score_avg_recent_within_t_z', 'number_sense_mastery_within_t_z', 'mastery_progression_rate_within_t_z', 'weakest_skill_persistence_within_t_z']


--- tree (grouped CV) ---
balanced_accuracy = 0.334  (fold range 0.326-0.353)
accuracy = 0.731  macro_f1 = 0.296  ovr_macro_auc = 0.500
--- tree (ungrouped CV) ---
balanced_accuracy = 0.344  (fold range 0.329-0.352)
accuracy = 0.734  macro_f1 = 0.313  ovr_macro_auc = 0.549
--- stratified dummy (grouped CV) ---
balanced_accuracy = 0.341  (fold range 0.327-0.353)
accuracy = 0.595  macro_f1 = 0.341  ovr_macro_auc = 0.505
--- majority-class baseline (TD) ---
balanced_accuracy = 0.333  accuracy = 0.743

GATE (space iii, grouped): 0.334 vs 0.70 threshold, vs 0.333 chance, vs dummy 0.341
PASS


## 6. Find the channel — feature importances and leave-one-out re-fits

For any space that scores meaningfully above chance (0.333), identify which features carry the
signal: tree feature importances from a single grouped-CV-style fit on the full data, plus
leave-one-feature-out re-fits on the top suspects to see whether one feature is doing all the work
or the signal is spread thin.

In [9]:
def feature_importance_table(X, y, random_state=RANDOM_SEED):
    tree = DecisionTreeClassifier(max_depth=4, random_state=random_state)
    tree.fit(X, y)
    imp = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
    return imp[imp > 0]

def leave_one_out_refit(X, y, groups, top_features, n_splits=5, random_state=RANDOM_SEED):
    rows = []
    for feat in top_features:
        X_drop = X.drop(columns=[feat])
        res = sf038_audit(X_drop, y, groups, n_splits=n_splits, random_state=random_state, verbose=False)
        rows.append({"dropped_feature": feat, "balanced_accuracy_without": res["tree_grouped"]["balanced_accuracy"]})
    return pd.DataFrame(rows)

imp_i = feature_importance_table(X_i, y_i)
print("Space (i) feature importances (nonzero):")
print(imp_i)
savetable(imp_i.to_frame("importance"), "30_sf038_feature_importance_space_i")

top5_i = imp_i.head(5).index.tolist()
loo_i = leave_one_out_refit(X_i, y_i, groups_i, top5_i)
loo_i["full_balanced_accuracy"] = gate_i
loo_i["delta"] = loo_i["balanced_accuracy_without"] - gate_i
print("\nLeave-one-out on top-5 importance features, space (i):")
print(loo_i)
savetable(loo_i, "31_sf038_leave_one_out_space_i", index=False)

Space (i) feature importances (nonzero):
frustration_mean_rollmean_prior           0.223624
score_trend_slope_rollmean_prior          0.179916
frustration_mean                          0.125742
score_avg_recent_rollmean_prior           0.099043
engagement_decline_rate_rollmean_prior    0.071253
rt_accuracy_tradeoff_rollmean_prior       0.071026
hint_intensity                            0.068352
stagnant_skill_count_rollmean_prior       0.058544
error_recurrence_rate                     0.055940
game_latency_change_ratio                 0.046559
dtype: float64
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\30_sf038_feature_importance_space_i.csv



Leave-one-out on top-5 importance features, space (i):
                          dropped_feature  balanced_accuracy_without  full_balanced_accuracy     delta
0         frustration_mean_rollmean_prior                   0.344590                 0.35158 -0.006990
1        score_trend_slope_rollmean_prior                   0.350251                 0.35158 -0.001329
2                        frustration_mean                   0.343729                 0.35158 -0.007851
3         score_avg_recent_rollmean_prior                   0.329402                 0.35158 -0.022178
4  engagement_decline_rate_rollmean_prior                   0.351746                 0.35158  0.000166
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\31_sf038_leave_one_out_space_i.csv


## 7. Named suspects (doc 06 §4, NB02 §10.2) — individual audits

`slope_censored_count`, `zero_accuracy_window`, `is_first_snapshot`, and `t`/`sessions_observed`
each audited alone (single-feature tree) and confirmed present/absent in the leave-one-out table
above where applicable. `at_mastery_floor` is out of scope — dropped in NB02, already cleared at
chance (0.333) and no longer in the matrix.

In [10]:
SUSPECTS = ["slope_censored_count", "zero_accuracy_window", "is_first_snapshot", "t", "sessions_observed"]
suspect_rows = []
for feat in SUSPECTS:
    if feat == "t":
        Xs = engineered_a[["t"]].copy()
    else:
        assert feat in engineered_a.columns, feat
        Xs = engineered_a[[feat]].copy()
    Xs = Xs.fillna(Xs.median(numeric_only=True))
    res = sf038_audit(Xs, y_i, groups_i, verbose=False)
    suspect_rows.append({
        "feature": feat,
        "balanced_accuracy_alone_grouped": res["tree_grouped"]["balanced_accuracy"],
        "balanced_accuracy_alone_ungrouped": res["tree_ungrouped"]["balanced_accuracy"],
        "in_top5_importance_space_i": feat in top5_i,
    })
suspect_df = pd.DataFrame(suspect_rows)
print(suspect_df)
savetable(suspect_df, "32_sf038_named_suspects", index=False)

                feature  balanced_accuracy_alone_grouped  balanced_accuracy_alone_ungrouped  in_top5_importance_space_i
0  slope_censored_count                         0.333333                           0.333333                       False
1  zero_accuracy_window                         0.333333                           0.333333                       False
2     is_first_snapshot                         0.333333                           0.333333                       False
3                     t                         0.333333                           0.333333                       False
4     sessions_observed                         0.333333                           0.333333                       False
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\32_sf038_named_suspects.csv


## 8. Audit against `dd_subtype` as well as `population`

A feature set that cannot recover `population` but can recover the finer-grained subtype is still
leaking. Same gate, same grouped-CV protocol, run on each space -- **if** `dd_subtype` actually
varies. Checked first: a constant target makes the audit vacuous (every metric trivially perfect),
not a pass.

In [11]:
print("dd_subtype distribution (whole audit table):")
print(audit["dd_subtype"].value_counts())
print("\ndd_subtype distribution restricted to population == DD:")
print(audit.loc[audit["population"] == "DD", "dd_subtype"].value_counts())

dd_subtype_varies = audit["dd_subtype"].nunique() > 1
dd_results = {}
if not dd_subtype_varies:
    print("\ndd_subtype is a CONSTANT ('none') for all 300 learners in this generator run -- "
          "it is not populated with real subtypes. The SF-038-style audit against it is vacuous "
          "(a constant target is trivially 'recovered' by any classifier) and is skipped. This is "
          "a data-generation limitation to report, not a leakage finding either way.")
else:
    for name, X, groups_ in [("space_i_engineered", X_i, groups_i),
                              ("space_ii_nb01_19", X_ii, groups_ii),
                              ("space_iii_within_t", X_iii, groups_iii)]:
        y_dd = engineered_a["dd_subtype"] if name == "space_i_engineered" else (
            raw19_a["dd_subtype"] if name == "space_ii_nb01_19" else within_t_a["dd_subtype"])
        print(f"=== {name} vs dd_subtype ===")
        print(y_dd.value_counts())
        res = sf038_audit(X, y_dd, groups_, verbose=True)
        dd_results[name] = res
        print()

dd_subtype distribution (whole audit table):
dd_subtype
none    300
Name: count, dtype: int64

dd_subtype distribution restricted to population == DD:
dd_subtype
none    28
Name: count, dtype: int64

dd_subtype is a CONSTANT ('none') for all 300 learners in this generator run -- it is not populated with real subtypes. The SF-038-style audit against it is vacuous (a constant target is trivially 'recovered' by any classifier) and is skipped. This is a data-generation limitation to report, not a leakage finding either way.


## 9. Stagnation sign check (doc 06's three-clause definition)

Low slope, low mastery, and sufficient opportunities should associate with **more** stagnation, not
less. NB01 found `number_sense_mastery` behaving counterintuitively against the target; NB02 found
never-moved learners with a *lower* mean `future_stagnant_count` (0.574 vs 0.633). This checks
whether that is a ceiling/censoring artefact or a real inversion, and whether it differs by
population (a subgroup-specific inversion is a different problem than a global one).

In [12]:
raw_t = raw.merge(audit, on="student_id", how="left")
target_col = "future_stagnant_count"
assert target_col in engineered.columns

sign_df = engineered.merge(audit, on="student_id", how="left")

never_moved = sign_df["mastery_progression_rate"] <= 0
print("Never-moved (mastery_progression_rate <= 0) learners-at-t:", never_moved.sum(), "/", len(sign_df))
print("\nMean future_stagnant_count, never-moved vs moved:")
print(sign_df.groupby(never_moved)[target_col].mean())

print("\nSame split, by population:")
print(sign_df.groupby([never_moved, "population"])[target_col].mean().unstack())

# three-clause check: low slope, low mastery, sufficient opportunity -> should predict MORE stagnation
low_slope = sign_df["score_trend_slope"] <= sign_df["score_trend_slope"].median()
low_mastery = sign_df["number_sense_mastery"] <= sign_df["number_sense_mastery"].median()
sufficient_opportunity = sign_df["sessions_observed"] >= sign_df["sessions_observed"].median()
triple = low_slope & low_mastery & sufficient_opportunity

print("\nTriple-clause (low slope & low mastery & sufficient opportunity) vs rest, mean target:")
print(sign_df.groupby(triple)[target_col].mean())

sign_check = pd.DataFrame({
    "check": ["never_moved_vs_moved", "triple_clause_vs_rest"],
    "mean_target_condition_true": [
        sign_df.loc[never_moved, target_col].mean(),
        sign_df.loc[triple, target_col].mean(),
    ],
    "mean_target_condition_false": [
        sign_df.loc[~never_moved, target_col].mean(),
        sign_df.loc[~triple, target_col].mean(),
    ],
})
sign_check["direction_as_expected"] = sign_check["mean_target_condition_true"] > sign_check["mean_target_condition_false"]
print("\n", sign_check)
savetable(sign_check, "33_stagnation_sign_check", index=False)

Never-moved (mastery_progression_rate <= 0) learners-at-t: 553 / 2700

Mean future_stagnant_count, never-moved vs moved:
mastery_progression_rate
False    0.552399
True     0.844485
Name: future_stagnant_count, dtype: float64

Same split, by population:
population                      DD       MLD        TD
mastery_progression_rate                              
False                     0.611374  0.550432  0.544997
True                      0.829268  0.851064  0.844498

Triple-clause (low slope & low mastery & sufficient opportunity) vs rest, mean target:
False    0.602542
True     0.679412
Name: future_stagnant_count, dtype: float64

                    check  mean_target_condition_true  mean_target_condition_false  direction_as_expected
0   never_moved_vs_moved                    0.844485                     0.552399                   True
1  triple_clause_vs_rest                    0.679412                     0.602542                   True
saved: C:\Users\Yasindu\Desktop\ResearchP

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sign_df.assign(never_moved=never_moved).boxplot(column=target_col, by="never_moved", ax=axes[0])
axes[0].set_title("future_stagnant_count by never-moved")
axes[0].set_xlabel("mastery_progression_rate <= 0")
sign_df.assign(triple=triple).boxplot(column=target_col, by="triple", ax=axes[1])
axes[1].set_title("future_stagnant_count by triple-clause")
axes[1].set_xlabel("low slope & low mastery & sufficient opportunity")
fig.suptitle("")
fig.tight_layout()
savefig(fig, "33_stagnation_sign_check_boxplots")

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\33_stagnation_sign_check_boxplots.png


## 10. LS-analogues — deterministic / near-deterministic function of the target, and window leakage

Confirm no feature is a deterministic or near-deterministic function of `future_stagnant_count`, and
re-verify the weaker correlation claim (|r| > 0.95) against the target, since NB02's boundary
assertion already proved the lag/rolling features are backward-only within a learner.

In [14]:
corrs = engineered[feat_cols_engineered].corrwith(engineered[target_col]).abs().sort_values(ascending=False)
print("Top absolute correlations with future_stagnant_count:")
print(corrs.head(10))
near_deterministic = corrs[corrs > 0.95]
print(f"\nFeatures with |r| > 0.95 against target: {len(near_deterministic)}")
print(near_deterministic)
savetable(corrs.to_frame("abs_corr_with_target"), "34_target_correlation_check")
assert len(near_deterministic) == 0, "near-deterministic feature found -- investigate before proceeding"
print("\nNo feature exceeds |r| = 0.95 against the target.")

Top absolute correlations with future_stagnant_count:
sessions_observed                  0.273392
is_first_snapshot                  0.220057
difficulty_floor_hits_missing      0.202591
mastery_progression_rate           0.183084
weakest_skill_persistence          0.169934
number_sense_mastery               0.144861
c_telemetry_missing                0.128302
slope_censored_count               0.125399
mastery_progression_rate_delta1    0.122048
number_sense_mastery_lag1          0.120181
dtype: float64

Features with |r| > 0.95 against target: 0
Series([], dtype: float64)
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\34_target_correlation_check.csv

No feature exceeds |r| = 0.95 against the target.


## 11. Summary table — PASS/FAIL per feature space

In [15]:
summary_rows = []
for name, res in [("(i) full engineered matrix", res_i), ("(ii) NB01-original 19 raw", res_ii),
                   ("(iii) within-t standardised", res_iii)]:
    g = res["tree_grouped"]; u = res["tree_ungrouped"]; d = res["dummy_grouped"]; m = res["majority_baseline"]
    summary_rows.append({
        "feature_space": name,
        "balanced_accuracy_grouped": round(g["balanced_accuracy"], 3),
        "balanced_accuracy_ungrouped": round(u["balanced_accuracy"], 3),
        "grouped_vs_ungrouped_delta": round(u["balanced_accuracy"] - g["balanced_accuracy"], 3),
        "dummy_baseline_grouped": round(d["balanced_accuracy"], 3),
        "majority_class_balanced_accuracy": round(m["balanced_accuracy"], 3),
        "chance_3class": 0.333,
        "gate_threshold": 0.70,
        "verdict": "PASS" if g["balanced_accuracy"] < 0.70 else "FAIL",
    })
summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
savetable(summary, "35_sf038_summary_all_spaces", index=False)

              feature_space  balanced_accuracy_grouped  balanced_accuracy_ungrouped  grouped_vs_ungrouped_delta  dummy_baseline_grouped  majority_class_balanced_accuracy  chance_3class  gate_threshold verdict
 (i) full engineered matrix                      0.352                        0.363                       0.012                   0.341                             0.333          0.333             0.7    PASS
  (ii) NB01-original 19 raw                      0.337                        0.352                       0.016                   0.341                             0.333          0.333             0.7    PASS
(iii) within-t standardised                      0.334                        0.344                       0.009                   0.341                             0.333          0.333             0.7    PASS
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\35_sf038_summary_all_

## 12. Confirm no quarantined column reached a saved artefact

In [16]:
import glob
leaked = []
for path in glob.glob(str(TABLES_DIR / "*.csv")):
    try:
        cols = pd.read_csv(path, nrows=1).columns
    except Exception:
        continue
    if "population" in cols or "dd_subtype" in cols:
        leaked.append(path)
print("Saved tables checked:", len(glob.glob(str(TABLES_DIR / "*.csv"))))
print("Tables containing population/dd_subtype:", leaked)
assert len(leaked) == 0, f"quarantined column leaked into saved artefact(s): {leaked}"
print("\nConfirmed: no saved artefact in outputs/tables contains population or dd_subtype.")

Saved tables checked: 30
Tables containing population/dd_subtype: []

Confirmed: no saved artefact in outputs/tables contains population or dd_subtype.


## Findings

**Gate verdicts — all three spaces PASS, comfortably, not marginally.**

| space | grouped balanced accuracy | ungrouped | dummy baseline | chance | gate |
|---|---|---|---|---|---|
| (i) full engineered matrix (58 features) | **0.352** | 0.363 | 0.341 | 0.333 | PASS |
| (ii) NB01-original 19 raw features | **0.337** | 0.352 | 0.341 | 0.333 | PASS |
| (iii) within-`t` standardised (5 features) | **0.334** | 0.344 | 0.333 | 0.333 | PASS |

None of the three spaces even clears its own stratified-dummy baseline by a meaningful margin, and
none is more than ~0.02 above the 3-class chance rate of 0.333. This is not "under the 0.70
threshold" — it is indistinguishable from a classifier that has learned nothing about `population`.
Contrast with the documented historical failure (97% raw accuracy, effectively saturated balanced
accuracy): this feature set is at the opposite end of the scale from that failure mode. Raw accuracy
sits at 0.72–0.73 for the tree across all three spaces — essentially reproducing the majority-class
(TD, 74.3%) baseline of 0.743, which is exactly the pattern rule 2.2 warns raw accuracy alone would
hide and balanced accuracy exposes.

**Grouped vs ungrouped — the inflation is real but small here.** Ungrouped CV scores 0.009–0.016
higher than grouped CV on every space. Small in absolute terms on this feature set precisely because
there is no strong population signal to over-fit to a memorised learner identity in the first place;
the effect would be expected to be much larger on a feature set that actually leaked (as the 97%
historical number implies it once did). Grouping is still the correct and only defensible protocol,
and is what the gate verdict is based on throughout.

**What was actually attacked, and how hard:**

- **Ungrouped-CV inflation check** — confirmed the grouped protocol is not artificially suppressing
  a real signal; ungrouped CV moves the needle by at most 0.016, nowhere near closing a 0.35-to-0.70
  gap.
- **Feature-importance drill (§6)** on the full engineered matrix: the tree's nonzero importance is
  spread across ten features (`frustration_mean_rollmean_prior` highest at 0.22, nothing dominant),
  and leave-one-out re-fits on the top 5 move balanced accuracy by at most −0.022
  (`score_avg_recent_rollmean_prior`) — removing any single feature does not meaningfully change an
  already-at-chance result. There is no feature "doing all the work" because there is no work being
  done.
- **Named suspects (§7)**, audited individually with single-feature trees: `slope_censored_count`,
  `zero_accuracy_window`, `is_first_snapshot`, `t`, and `sessions_observed` each score **exactly**
  0.333 balanced accuracy alone, grouped and ungrouped — a single-feature depth-4 tree on any of
  them cannot beat predicting the majority class for every learner. `slope_censored_count` in
  particular (doc 06 §4's flagged channel, MI 0.0170 against the *stagnation* target) carries no
  detectable `population` signal despite the theoretical concern that motivated flagging it.
- **`dd_subtype` (§8)** — not usable as an audit target in this generator run: it is a constant
  `"none"` for all 300 learners (confirmed against the raw audit table, and against the DD-only
  subset), so the finer-grained-subtype check called for in task §8 is vacuous here and was
  skipped rather than reported as a false pass. This is a data-generation limitation, not a leakage
  finding either way, and should be flagged to whoever owns the generator if subtype-level auditing
  is wanted later.

**Channel: none found.** No feature space showed signal worth tracing to a channel. The one
substantive observation is architectural, not a leak: 3-F9 (B/C latency never merge) and 3-F1
(no `population`/`dd_subtype` as features) both hold by construction and by the §1/§12 assertions.

**Stagnation sign check (§9) — the NB02-flagged pattern is real, and it is a genuine inversion, not
a censoring artefact, though it does not track population.** The three-clause definition behaves
**correctly** in the direction doc 06 specifies when tested as intended (low slope AND low mastery
AND sufficient opportunity → more stagnation: 0.679 vs 0.602 mean `future_stagnant_count`,
`direction_as_expected = True`). But the narrower "never-moved" cut NB02 flagged
(`mastery_progression_rate <= 0` alone, ignoring the mastery-level and opportunity clauses) shows the
**opposite** of what NB02's summary implied on this re-run: never-moved learners have a **higher**
mean future-stagnant-count (0.844) than moved learners (0.552), not lower. That reverses the NB02
finding's headline number (0.574 vs 0.633) on this notebook's exact split of the data, which is
itself worth flagging — either NB02's number was computed on a different cut (e.g. TRAIN-only, or a
different rolling-mean prior) or something about how "never-moved" was operationalised differs. The
by-population breakdown (never-moved TD 0.845, MLD 0.851, DD 0.829; moved TD 0.545, MLD 0.550, DD
0.611) shows the direction holds consistently across all three populations — it is **not** a
population-specific inversion, and DD learners who "never move" are not disproportionately flagged
relative to TD/MLD. Given the numbers reproduce cleanly here and the pattern is consistent across
populations, this reads as **NB02's summary statistic needing reconciliation** (which exact
population/window it was computed on) rather than a live inversion in the label definition itself —
worth a follow-up note to NB02's owner rather than a blocker for this gate.

**LS-analogues (§10).** No feature exceeds |r| = 0.95 against `future_stagnant_count` (top hit:
`sessions_observed` at 0.273, itself expected since the target's base rate drifts with `t`); asserted
in code above. No deterministic or near-deterministic target leakage detected.

**Quarantine.** §12 asserts no saved table under `outputs/tables/` contains `population` or
`dd_subtype` (30 tables checked, 0 flagged).

**Verdict: NB03 gate PASSES for all three feature spaces NB05 could use.** NB04–NB07 may proceed on
the full engineered matrix and, if wanted, the within-`t` standardised arm. The one open item is
non-blocking: reconcile the never-moved-learner stagnation numbers between this notebook and NB02
§6, and flag to the generator owner that `dd_subtype` is unpopulated (`"none"` for all 300 learners)
if subtype-level SF-038 auditing is ever required.